# Tennis Match Data: A Friendly Tour

A plain-language tour of the tennis match data behind this project. We open
a local, read-only copy of the data and explore it with classic pandas
tools: `head()` for a peek, `info()` for sizes and missing values, and
`describe()` for summary numbers.

Here is what each part covers:

1. Meet the data - what each table looks like
2. A broad look at the data - dates, surfaces, and completeness
3. Player and match statistics
4. Comparisons: surfaces, tournaments, and the ranking gap
5. Player profiles - who the players are
6. Trends over time
7. Which columns later feed predictions

Two small conventions keep the charts honest:

- **Row grain**: each chart states its unit via `GRAINS` - `physical_match`
  (one row per match, for event counts), `player_match` (one player's point
  of view), `gold_directional` (one row per player per match), or the
  one-row-per-player profile frames. Gold stores each match twice (once per
  player), so event counts use the physical-match view to avoid double
  counting.
- **Sampling**: summaries and averages always use all rows. Only the very
  dense per-row scatter plots use a fixed random sample of the most recent
  10 years (20,000 rows), and every such chart says so.

Everything here is an observation about this data, not a claim about cause
and effect. Missing values mean "not recorded", not "something wrong".


## Setup: how we load the data

All analysis reads a local, read-only copy of the data from
`data/training_snapshot.duckdb` at the repository root. Nothing is written,
and no live database is touched.

The tables we work with:

| Frame | Source | What it holds |
|---|---|---|
| `gold` | `gold.match_features` | One row per player per match, with the outcome and stats comparing the two players |
| `silver` | `silver.player_matches` | One row per player per match, from that player's point of view |
| `matches` | `gold`, each match counted once | One row per physical match |
| `gold_profiles` | `gold.player_profiles` | One row per player: career summary numbers |
| `bronze_profiles` | `bronze.player_profiles` | One row per player: biographical facts (height, handedness, ...) |

- Serve/return rates are worked out in the notebook from `silver` counts,
  with zero denominators mapped to NaN (a "not available" marker).
- Dense charts sample the most recent 10 years with a fixed seed into
  `*_10y_sample` frames. Summaries and averages always use all rows.
- Every chart declares its grain by picking a frame from `GRAINS`.

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.constants import ROOT

# Read-only handle to the local data snapshot.
SNAPSHOT_PATH = ROOT / "data" / "training_snapshot.duckdb"

con = duckdb.connect(str(SNAPSHOT_PATH), read_only=True)

sns.set_theme(style="whitegrid", palette="deep")

In [ ]:
# Gold directional features: one row per player per match (reciprocal pairs).
gold = con.table("gold.match_features").to_df()

# Player-perspective matches with serve/return counts.
silver = con.table("silver.player_matches").to_df()

# One row per player: aggregated metrics (gold) and raw bio fields (bronze).
gold_profiles = con.table("gold.player_profiles").to_df()
bronze_profiles = con.table("bronze.player_profiles").to_df()

con.close()

## Meet the data

Now that everything is loaded, let's look at each table with the classic
pandas moves: `head()` for a peek at the rows, `info()` for the size and
missing values, and `describe()` for summary numbers.

Quick guide to the four tables:

- **`gold`** - every match from each player's side: who played, when, on
  what surface, who won, and stats that compare the two players (for
  example `rank_diff` = the player's rank minus the opponent's rank).
- **`silver`** - the same matches from a player's point of view, including
  the raw serve/return counts (aces, double faults, points won, ...).
- **`gold_profiles`** - one row per player with career summary numbers
  (matches played, serve percentage, win rate, current rank).
- **`bronze_profiles`** - one row per player with biographical facts:
  birth date, height, weight, when they turned pro, and handedness.

Run each cell below and look at the output - the point is just to get a
feel for what the data looks like.


In [ ]:
# gold: one row per player per match. info() shows the row count, the
# columns, and how many values are missing in each column.
gold.info()
print()
gold.head()

In [ ]:
# silver: one row per player per match, with the raw serve/return counts.
silver.info()
print()
silver.head()

In [ ]:
# gold_profiles: one row per player, with career summary numbers.
gold_profiles.info()
print()
gold_profiles.head()
print()
gold_profiles.describe()

In [ ]:
# bronze_profiles: one row per player, with biographical facts. describe
# with include="all" also summarizes the text column (handedness). Many
# values are missing here - that is normal, not an error.
bronze_profiles.info()
print()
bronze_profiles.head()
print()
bronze_profiles.describe(include="all")

In [ ]:
# A quick summary: how many rows in each table, over which dates, and on
# which surfaces. Gold stores each match twice (once per player), so the
# surface counts below count each match only once.
print("Rows in each table:")
print(
    pd.Series(
        {
            "gold (player-match rows)": len(gold),
            "silver (player-match rows)": len(silver),
            "gold_profiles (players)": len(gold_profiles),
            "bronze_profiles (players)": len(bronze_profiles),
        }
    ).to_string()
)
print()
print(f"Matches span {gold['match_date'].min():%Y-%m-%d} to {gold['match_date'].max():%Y-%m-%d}")
print()
print("Matches per surface (each match counted once):")
print(gold.drop_duplicates("match_id")["surface"].value_counts().to_string())

What to notice here:

- **Size** - the tables are big: tens of thousands of player-match rows and
  a few thousand players.
- **Dates** - the data goes back decades, so older matches have fewer
  recorded statistics.
- **Surfaces** - hard courts dominate, and clay and grass make up most of
  the rest.
- **Missing values** - `info()` shows the NaN counts. In the player profiles
  especially, many values are simply not recorded - a fact about the data,
  not a mistake.


In [ ]:
# One row per physical match for event counts; gold holds two directional rows
# per match, so deduplication halves the frame.
matches = gold.drop_duplicates("match_id").sort_values(["match_date", "match_id"])

In [ ]:
# Safe in-memory serve/return rates from silver counts; zero denominators become NaN.
silver["second_serves"] = silver["total_serve_points"] - silver["first_serves_made"]
silver["points_won_on_serve"] = silver["first_serve_points_won"] + silver["second_serve_points_won"]

SERVE_RATE_COLS = {
    "ace_rate": ("aces", "total_serve_points"),
    "df_rate": ("double_faults", "total_serve_points"),
    "first_serve_pct": ("first_serves_made", "total_serve_points"),
    "first_serve_win_pct": ("first_serve_points_won", "first_serves_made"),
    "second_serve_win_pct": ("second_serve_points_won", "second_serves"),
    "serve_win_pct": ("points_won_on_serve", "total_serve_points"),
    "return_points_won_pct": ("return_points_won", "return_points_available"),
    "break_points_saved_pct": ("break_points_saved", "break_points_faced"),
    "aces_per_svc_game": ("aces", "service_games"),
}
for col, (numerator, denominator) in SERVE_RATE_COLS.items():
    silver[col] = silver[numerator] / silver[denominator].replace(0, np.nan)

In [ ]:
# Reproducible dense-chart sample: most recent 10 years, fixed seed, capped size.
# The full gold/silver frames above are never sampled; only these views are.
LAST_YEARS = 10
RNG_SEED = 42
DENSE_SAMPLE_N = 20_000

last_10y_start = gold["match_date"].max() - pd.DateOffset(years=LAST_YEARS)
gold_10y = gold[gold["match_date"] >= last_10y_start]
silver_10y = silver[silver["match_date"] >= last_10y_start]
gold_10y_sample = gold_10y.sample(n=min(DENSE_SAMPLE_N, len(gold_10y)), random_state=RNG_SEED)
silver_10y_sample = silver_10y.sample(n=min(DENSE_SAMPLE_N, len(silver_10y)), random_state=RNG_SEED)

Chart cells start by declaring their grain, e.g. `df = GRAINS["physical_match"]`.
Counts use `physical_match`; player statistics use `player_match`; target and
feature diagnostics use `gold_directional`; dense plots use the fixed-seed
`last_10y_*_sample` frames; profiles use the one-row-per-player frames.

In [ ]:
# Declare the data grain per chart; summaries/correlations always use the full
# frames, dense plots the fixed-seed 10-year samples.
GRAINS = {
    "physical_match": matches,
    "player_match": silver,
    "gold_directional": gold,
    "gold_profiles": gold_profiles,
    "bronze_profiles": bronze_profiles,
    "last_10y_gold_sample": gold_10y_sample,
    "last_10y_silver_sample": silver_10y_sample,
}

## A broad look at the data

Before diving into tennis questions, this section shows how much data we
have, over which dates, on which surfaces, and where values are missing.
Missing values here mean "not recorded", not "something went wrong" - for
example, serve statistics were not recorded before the 1990s. Event counts
use one row per physical match so each match is counted once.


What to look for: whether the date range and surface coverage match ATP history, how much of the data predates serve-stat recording, and which fields are too sparse to trust in later charts.

In [ ]:
# Data-context summary: sizes, date coverage, reciprocal structure, categorical
# coverage. Aggregate context uses the full frames, never the samples.
frames = {
    "gold (directional)": gold,
    "silver (player match)": silver,
    "physical match": matches,
    "gold profiles": gold_profiles,
    "bronze profiles": bronze_profiles,
}
print(pd.Series({name: len(frame) for name, frame in frames.items()}, name="rows").to_string())
print()
print(
    f"match_date coverage: {matches['match_date'].min():%Y-%m-%d} to {matches['match_date'].max():%Y-%m-%d}"
)

# Reciprocal structure: gold stores two directional rows per physical match, so
# this count shows the directional row shape behind the grains.
rows_per_match = gold.groupby("match_id").size()
print(f"directional rows per match_id: {rows_per_match.value_counts().to_dict()}")

for col in ["surface", "round", "tournament_level"]:
    counts = matches[col].value_counts(dropna=False)
    print(f"\n{col} (n={counts.size}, nulls={matches[col].isna().sum()}):")
    print(counts.to_string())

In [ ]:
# Missingness for plotting-relevant fields. Raw serve counts are never null, but
# serve rates are undefined when a row has zero recorded serve points, so that
# share is reported separately as unavailable serve exposure.
missing_fields = {
    "player_ranking (silver)": silver["player_ranking"].isna().mean(),
    "player_age (silver)": silver["player_age"].isna().mean(),
    "serve exposure (silver, zero serve points)": (silver["total_serve_points"] == 0).mean(),
    "serve pct (gold_profiles)": gold_profiles["first_serve_in_pct"].isna().mean(),
    "current_rank (gold_profiles)": gold_profiles["current_rank"].isna().mean(),
    "handedness (bronze_profiles)": bronze_profiles["handedness"].isna().mean(),
    "height (bronze_profiles)": bronze_profiles["height"].isna().mean(),
    "weight (bronze_profiles)": bronze_profiles["weight"].isna().mean(),
    "birthdate (bronze_profiles)": bronze_profiles["birthdate"].isna().mean(),
    "turned_pro (bronze_profiles)": bronze_profiles["turned_pro"].isna().mean(),
}
missing = pd.Series(missing_fields, name="share_unavailable").sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(
    data=missing.reset_index().rename(columns={"index": "field"}),
    x="share_unavailable",
    y="field",
    ax=ax,
)
ax.set_title("Share of rows with unavailable values (data availability, not a tennis pattern)")
ax.set_xlabel("share of rows without the value")
ax.set_ylabel("")
fig.tight_layout()

# Grain: player_match (silver), full frame. Serve stats were not recorded for
# essentially all pre-1990 rows, so serve/return analyses are only meaningful later.
by_year = silver.assign(year=silver["match_date"].dt.year).groupby("year")["total_serve_points"]
serve_unavailable = by_year.apply(lambda s: (s == 0).mean())

fig, ax = plt.subplots(figsize=(11, 3.5))
sns.lineplot(x=serve_unavailable.index, y=serve_unavailable.values, ax=ax)
ax.set_title(
    "Share of player-match rows with no serve stats by year (player_match grain, full frame)"
)
ax.set_xlabel("year")
ax.set_ylabel("share without serve stats")
fig.tight_layout()

In [ ]:
# Grain: physical_match (one row per match_id), full frame; aggregate, never sampled.
df = GRAINS["physical_match"]
year_counts = df["match_date"].dt.year.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(11, 3.5))
sns.lineplot(x=year_counts.index, y=year_counts.values, ax=ax)
ax.set_title(
    f"Physical match volume per year (n={len(df):,} matches, one row per match_id, full frame)"
)
ax.set_xlabel("year")
ax.set_ylabel("matches")
fig.tight_layout()

In [ ]:
# Grain: physical_match, full frame. Carpet disappears from the data after 2017,
# a coverage fact (surface availability), not a performance pattern.
df = GRAINS["physical_match"]
surface_year = (
    df.assign(year=df["match_date"].dt.year)
    .groupby(["year", "surface"])
    .size()
    .reset_index(name="count")
)

fig, ax = plt.subplots(figsize=(11, 3.5))
sns.lineplot(data=surface_year, x="year", y="count", hue="surface", ax=ax)
ax.set_title("Surface composition by year (physical_match grain, full frame)")
ax.set_xlabel("year")
ax.set_ylabel("matches")
fig.tight_layout()

In [ ]:
# Grain: physical_match, full frame. tournament_level is the gold numeric encoding;
# round keeps non-standard labels (e.g. 3rd/4th, r256) so coverage is not hidden.
df = GRAINS["physical_match"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(
    data=df, x="tournament_level", order=sorted(df["tournament_level"].unique()), ax=axes[0]
)
axes[0].set_title("Matches by tournament_level (gold encoding)")
axes[0].set_xlabel("tournament_level")
axes[0].set_ylabel("matches")

round_order = df["round"].value_counts().index
sns.countplot(data=df, x="round", order=round_order, ax=axes[1])
axes[1].set_title("Matches by round (non-standard labels kept)")
axes[1].set_xlabel("round")
axes[1].set_ylabel("matches")
axes[1].tick_params(axis="x", rotation=45)
fig.tight_layout()

## Player and match statistics

Distribution of the per-player and per-match quantities we explore: rank,
age, career match volume, recent workload, and the observed serve/return rates
derived from silver counts. A few simple policies keep the charts honest:

- **Grain and coverage**: every chart declares its grain via `GRAINS`. Dense
  per-row charts (rank, age, workload, rates) use the fixed-seed
  `last_10y_silver_sample`; the one-row-per-player volume chart uses the full
  `gold_profiles` frame (~7.6k rows, not dense).
- **Safe denominators**: rates divide by recorded exposure (serve points,
  service games, break points faced) with zero denominators mapped to NaN in
  t1-05. Rows without serve exposure (~4% of the 10y window, walkovers and
  retirements) are dropped here; the pre-1990 absence of serve stats was
  quantified in the previous section.
- **Excluded values**: probability rates outside [0, 1] and ages outside the
  playing career [15, 50] are dropped and counted per chart so the plotted
  distributions stay interpretable; nothing is clipped into view.
- **Tails**: rank and career match volume use log-x or clipped views so the
  high-volume bulk is not crushed by the long tail.

What to look for: where the bulk of players sit in rank and age, how concentrated career match volume is, and which serve/return rates have wide or multi-modal spreads - the raw material the later prediction step reads.

In [ ]:
# Grain: player_match, last_10y_silver_sample (dense). Rows without a ranking
# (1.6% of the window, mostly unranked qualifiers) are excluded and counted.
# Rank is roughly log-distributed to 2159, so the linear view hides the bulk
# and the log-x view restores it.
df = GRAINS["last_10y_silver_sample"]
ranked = df["player_ranking"].dropna()
unranked = df["player_ranking"].isna().sum()

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4))
sns.histplot(ranked, bins=60, ax=axes[0])
axes[0].set_title(f"player_ranking, linear (n={len(ranked):,}; {unranked} unranked rows excluded)")
axes[0].set_xlabel("ranking")
axes[0].set_ylabel("player-match rows")

sns.histplot(ranked, bins=60, log_scale=True, ax=axes[1])
axes[1].set_title("player_ranking, log x (bulk of the field sits far above rank 1)")
axes[1].set_xlabel("ranking (log)")
axes[1].set_ylabel("player-match rows")
fig.tight_layout()

In [ ]:
# Grain: player_match, last_10y_sample (dense). Ages are kept to the plausible
# playing career [15, 50]; missing and out-of-range values are dropped and
# counted, not clipped into view.
df = GRAINS["last_10y_silver_sample"]
raw_age = df["player_age"].dropna()
age = raw_age[(raw_age >= 15) & (raw_age <= 50)]
dropped = len(df) - len(age)

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(age, bins=40, ax=ax)
ax.set_title(f"player_age at match time (n={len(age):,}; {dropped} missing/out-of-range excluded)")
ax.set_xlabel("age (years)")
ax.set_ylabel("player-match rows")
fig.tight_layout()

In [ ]:
# Grain: gold_profiles, full frame (one row per player, ~7.6k rows, not dense).
# Career match volume is power-law with a long tail (max 1586), so log-x shows
# the bulk. The 13 players with zero recorded matches are excluded and counted.
df = GRAINS["gold_profiles"]
volume = df["match_count"]
volume = volume[volume >= 1]
zero_match_players = (df["match_count"] < 1).sum()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(volume, bins=50, log_scale=True, ax=ax)
ax.set_title(
    f"career matches per player, log x (n={len(volume):,} players, "
    f"median={volume.median():.0f}; {zero_match_players} with 0 matches excluded)"
)
ax.set_xlabel("career matches (log)")
ax.set_ylabel("players")
fig.tight_layout()

In [ ]:
# Grain: player_match, last_10y_sample (dense). matches_30d_before is a small
# integer workload count (0-22), so a count plot needs no binning or scaling.
df = GRAINS["last_10y_silver_sample"]
workload = df["matches_30d_before"]

fig, ax = plt.subplots(figsize=(9, 4))
sns.countplot(data=df, x="matches_30d_before", ax=ax)
ax.set_title(
    f"matches in the 30 days before each match (n={len(workload):,} "
    f"player-match rows; median={workload.median():.0f})"
)
ax.set_xlabel("matches_30d_before")
ax.set_ylabel("player-match rows")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()

In [ ]:
# Grain: player_match, last_10y_sample (dense). Rates come from silver counts
# with zero denominators already NaN (t1-05); rows without serve exposure are
# dropped (~4% of the window). Probability rates outside [0, 1] are excluded so
# the plotted distribution stays interpretable; aces_per_svc_game is a per-game
# count, not a probability, so multi-ace games above 1 are legitimate.
df = GRAINS["last_10y_silver_sample"]
PROB_RATES = set(SERVE_RATE_COLS) - {"aces_per_svc_game"}

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
fig.suptitle(
    "Observed serve/return rates from silver counts (zero-exposure and out-of-range rows excluded)"
)
for ax, col in zip(axes.ravel(), SERVE_RATE_COLS, strict=True):
    s = df[col].dropna()
    if col in PROB_RATES:
        s = s[(s >= 0) & (s <= 1)]
    sns.histplot(s, bins=40, ax=ax)
    ax.set_title(f"{col}\n(n={len(s):,}, {100 * len(s) / len(df):.0f}% of rows)")
    ax.set_xlabel("")
    ax.set_ylabel("")
fig.tight_layout(rect=(0, 0, 1, 0.97))

## Comparisons: surfaces, tournaments, and the ranking gap

Observational tennis comparisons over the most recent 10 years of the data:

- **Window**: this section uses the last 10 years only. Aggregates (box/violin/bar
  and binned summaries) use the full 10-year window; dense per-row scatters use the
  fixed-seed `last_10y_*_sample` frames (20k capped, seed 42).
- **Units**: observed serve/return performance uses `player_match` rows; event
  proportions (win rates, rank-gap win probability) use one row per physical match.
- **Exclusions**: rows without serve exposure or with out-of-range rates are dropped
  per chart and counted; carpet is excluded where its window sample is tiny
  (availability, not a pattern).
- **Interpretation**: every comparison here is observational. A difference between
  surfaces, levels, or players describes this data; it does not establish causality.

What to look for: whether serve/return metrics separate by surface in this data, how often the better-ranked player wins overall and by gap and level, and whether surface specialization is visible for the most active players. Differences describe this data; they do not prove causation.

In [ ]:
# Extend GRAINS with the full (unsampled) last-10-year window for aggregate charts.
# Binned and bar summaries use these; only dense per-row scatters use the fixed-seed
# 20k samples defined in t1-06.
GRAINS["last_10y_silver"] = silver_10y
GRAINS["last_10y_physical"] = gold_10y.drop_duplicates("match_id").sort_values(
    ["match_date", "match_id"]
)

In [ ]:
# Grain: player_match, full last-10y window (aggregate, never sampled). Observed
# serve/return performance from silver counts (t1-05). Rows without serve exposure
# and out-of-range probability rates are dropped per metric and counted. Carpet is
# excluded: 7 player-match rows in the window, a data-availability fact. All
# comparisons are observational, not causal.
SURFACES = ["hard", "clay", "grass"]
SURFACE_COLS = ["ace_rate", "first_serve_pct", "first_serve_win_pct", "return_points_won_pct"]
df = GRAINS["last_10y_silver"]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.ravel(), SURFACE_COLS, strict=True):
    s = df.dropna(subset=[col])
    s = s[(s[col] >= 0) & (s[col] <= 1) & s["surface"].isin(SURFACES)]
    dropped = len(df) - len(s)
    sns.violinplot(
        data=s,
        x="surface",
        y=col,
        order=SURFACES,
        hue="surface",
        hue_order=SURFACES,
        inner="box",
        legend=False,
        ax=ax,
    )
    ax.set_title(f"{col} by surface (n={len(s):,})")
    ax.set_xlabel("")
    print(f"{col}: {len(s):,} rows kept, {dropped:,} excluded (no serve exposure / out-of-range)")
fig.suptitle(
    "Serve/return metrics by surface - player_match grain, full last-10y window (observational)",
    y=0.97,
)
fig.tight_layout(rect=(0, 0, 1, 0.9))

In [ ]:
# Grain: physical_match (one row per match_id), full last-10y window. Win rate is an
# event proportion, so the unit is the physical match. "Favorite" = the better-ranked
# player (rank_diff = player_ranking - opponent_ranking < 0). Carpet is excluded:
# 3 matches in the window (availability). Observational, not causal.
df = GRAINS["last_10y_physical"]
fav = df[(df["rank_diff"] < 0) & df["surface"].isin(SURFACES)].copy()
overall = fav["match_won"].mean()
print(
    f"favorite win rate overall: {overall:.3f} "
    f"(n={len(fav):,} physical matches, full last-10y window)"
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, groupby in zip(axes, ["surface", "tournament_level"], strict=True):
    agg = (
        fav.groupby(groupby, observed=True)["match_won"]
        .agg(["mean", "count"])
        .sort_values("count", ascending=False)
    )
    order = agg.index.tolist()
    sns.barplot(data=fav, x=groupby, y="match_won", order=order, ax=ax)
    ax.axhline(overall, color="gray", ls="--", lw=1)
    ax.text(len(order) - 0.4, overall + 0.012, f"overall {overall:.2f}", fontsize=8, color="gray")
    for bar, n in zip(ax.patches, agg["count"], strict=True):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"n={n:,}",
            ha="center",
            fontsize=8,
        )
    ax.set_ylim(0, 0.8)
    ax.set_title(f"Better-ranked player win rate by {groupby}")
    ax.set_ylabel("observed win probability")
    ax.set_xlabel(groupby if groupby == "surface" else "tournament_level (gold encoding)")
fig.suptitle(
    "Favorite win rate - physical_match grain, full last-10y window (observational)",
    y=0.97,
)
fig.tight_layout(rect=(0, 0, 1, 0.86))

In [ ]:
# Grain: player_match, full last-10y window. Player surface specialization for a
# limited, clearly qualified set: the 8 most active players who played >= 400 matches
# in the window and >= 10 on each of hard/clay/grass, so no one-match surface record
# can dominate. Bars are observed ace_rate / first_serve_pct; per-surface sample
# sizes are annotated on the left panel and printed below.
MIN_TOTAL, MIN_PER_SURFACE = 400, 10
df = GRAINS["last_10y_silver"]
totals = df.groupby("player_id").size()
cand = totals[totals >= MIN_TOTAL].index
surf_counts = (
    df[df["player_id"].isin(cand)].groupby(["player_id", "surface"]).size().unstack(fill_value=0)
)
qualified = surf_counts[(surf_counts[["hard", "clay", "grass"]] >= MIN_PER_SURFACE).all(axis=1)]
qualified = qualified.reindex(totals[qualified.index].sort_values(ascending=False).index).head(8)
players = qualified.index.tolist()

sub = df[df["player_id"].isin(players)]
sub = sub.dropna(subset=["ace_rate", "first_serve_pct"])
sub = sub[(sub["ace_rate"] <= 1) & (sub["first_serve_pct"] <= 1)]
means = (
    sub.groupby(["player_id", "surface"], observed=True)[["ace_rate", "first_serve_pct"]]
    .mean()
    .reset_index()
)
n_table = (
    sub.groupby(["player_id", "surface"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(players)[SURFACES]
)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, col in zip(axes, ["ace_rate", "first_serve_pct"], strict=True):
    sns.barplot(
        data=means, x="player_id", y=col, hue="surface", hue_order=SURFACES, order=players, ax=ax
    )
    ax.set_title(f"{col} per surface (n per bar on left panel)")
    ax.set_xlabel("player_id (activity-qualified)")
    ax.set_ylabel(col)
    ax.legend(title="surface")
    if col == "ace_rate":
        # Barplot patches are grouped by hue (all hard bars, then clay, then grass),
        # so transpose n_table to match that order; seaborn also appends empty bars.
        for bar, n in zip(ax.patches, n_table.to_numpy().T.ravel(), strict=False):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.002,
                f"{n}",
                ha="center",
                fontsize=6,
            )
        ax.set_ylim(0, means["ace_rate"].max() * 1.25)
fig.suptitle(
    "Player surface specialization - player_match, full last-10y window; "
    f"threshold >= {MIN_TOTAL} matches total and >= {MIN_PER_SURFACE} per surface (observational)",
    y=0.97,
)
print("Per-surface sample sizes (hard/clay/grass):")
print(n_table.to_string())
fig.tight_layout(rect=(0, 0, 1, 0.86))

In [ ]:
# Grain: physical_match (event unit), full last-10y window, binned aggregate (never
# sampled). Observed win probability of the better-ranked player by ranking-gap bin;
# counts are annotated so small bins stay visible. Observational: this is the
# realized outcome in this snapshot, not a guarantee or a causal claim.
df = GRAINS["last_10y_physical"]
fav = df[(df["rank_diff"] < 0) & df["surface"].isin(SURFACES)].copy()
fav["gap"] = -fav["rank_diff"]
GAP_BINS = [0, 20, 50, 100, 200, 400, np.inf]
GAP_LABELS = ["0-20", "20-50", "50-100", "100-200", "200-400", "400+"]
fav["gap_bin"] = pd.cut(fav["gap"], bins=GAP_BINS, labels=GAP_LABELS, right=False)
agg = fav.groupby("gap_bin", observed=True)["match_won"].agg(["mean", "count"])
overall = fav["match_won"].mean()
print(
    f"favorite win rate overall: {overall:.3f} (n={len(fav):,} physical matches, "
    "full last-10y window, binned aggregate)"
)

fig, ax = plt.subplots(figsize=(9.5, 5))
sns.barplot(x=agg.index, y=agg["mean"], ax=ax)
ax.axhline(overall, color="gray", ls="--", lw=1)
ax.text(len(agg) - 0.4, overall + 0.012, f"overall {overall:.2f}", fontsize=8, color="gray")
for bar, n in zip(ax.patches, agg["count"], strict=True):
    assert isinstance(bar, plt.Rectangle)
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"n={n:,}",
        ha="center",
        fontsize=8,
    )
ax.set_ylim(0, 0.85)
ax.set_title("Better-ranked player win probability by ranking gap (binned)")
ax.set_xlabel("ranking gap (better-ranked player's rank minus worse-ranked player's rank)")
ax.set_ylabel("observed win probability")
fig.tight_layout(rect=(0, 0, 1, 0.93))

In [ ]:
# Grain: player_match, fixed-seed last-10y sample (20k capped, seed 42) - dense
# scatter, so the sample policy applies. Age kept to the plausible career [15, 50];
# unranked rows are excluded and counted on the ranking panel. Ticks are pinned to
# the data range so no out-of-range tick labels overhang the figure. Observational,
# not causal.
import matplotlib.ticker

df = GRAINS["last_10y_silver_sample"]
d = df[df["player_age"].between(15, 50) & df["surface"].isin(SURFACES)].copy()
print(f"dense sample: {len(d):,} player-match rows (fixed-seed last-10y sample, 20k capped)")

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.2))
sns.scatterplot(
    data=d,
    x="player_age",
    y="matches_30d_before",
    hue="surface",
    hue_order=SURFACES,
    s=10,
    alpha=0.35,
    ax=axes[0],
)
axes[0].set_title("Recent workload by age")
axes[0].set_xlabel("player_age (years)")
axes[0].set_ylabel("matches_30d_before")
axes[0].set_xticks(range(15, 55, 5))

ranked = d.dropna(subset=["player_ranking"])
sns.scatterplot(
    data=ranked,
    x="player_ranking",
    y="matches_30d_before",
    hue="surface",
    hue_order=SURFACES,
    s=10,
    alpha=0.35,
    legend=False,
    ax=axes[1],
)
rank_max = ranked["player_ranking"].max()
axes[1].set_xscale("log")
axes[1].set_xlim(1, rank_max * 1.05)
axes[1].set_xticks([10**k for k in range(int(np.floor(np.log10(rank_max))) + 1)])
axes[1].xaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
axes[1].set_title(
    f"Recent workload by ranking (n={len(ranked):,}, {len(d) - len(ranked):,} unranked excluded)"
)
axes[1].set_xlabel("player_ranking (log)")
axes[1].set_ylabel("matches_30d_before")
fig.suptitle(
    "Recent workload by age/ranking, hue by surface - last-10y fixed-seed sample (observational)",
    y=0.97,
)
fig.tight_layout(rect=(0, 0, 0.97, 0.86))

In [ ]:
# Grain: player_match, fixed-seed last-10y sample (20k capped, seed 42) - dense
# scatter, so the sample policy applies. Rows without serve or return exposure and
# out-of-range rates are excluded. Ticks are pinned to the [0, 1] rate range so no
# out-of-range tick labels overhang the figure. A positive correlation means good
# servers also tend to win more return points in these rows; observational, not
# causal.
df = GRAINS["last_10y_silver_sample"]
d = df.dropna(subset=["serve_win_pct", "return_points_won_pct"])
d = d[(d["serve_win_pct"] <= 1) & (d["return_points_won_pct"] <= 1) & d["surface"].isin(SURFACES)]
r_overall = d["serve_win_pct"].corr(d["return_points_won_pct"])
print(
    f"dense sample: {len(d):,} rows with valid serve/return rates; "
    f"overall Pearson r={r_overall:.2f}"
)

tick_rates = np.arange(0, 1.01, 0.2)
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))
sns.regplot(
    data=d,
    x="serve_win_pct",
    y="return_points_won_pct",
    ax=axes[0],
    scatter_kws={"s": 8, "alpha": 0.15},
    line_kws={"color": "crimson"},
)
axes[0].set_title(
    f"Serve vs return win rates (n={len(d):,}, r={r_overall:.2f})\n"
    "player_match, last-10y fixed-seed sample"
)
axes[0].set_xlabel("serve_win_pct (observed)")
axes[0].set_ylabel("return_points_won_pct (observed)")
axes[0].set_xticks(tick_rates)
axes[0].set_yticks(tick_rates)

pal = sns.color_palette("deep", 3)
for surface, color in zip(SURFACES, pal, strict=True):
    sub = d[d["surface"] == surface]
    sns.regplot(
        data=sub,
        x="serve_win_pct",
        y="return_points_won_pct",
        ax=axes[1],
        color=color,
        scatter_kws={"s": 8, "alpha": 0.12},
        label=f"{surface} r={sub['serve_win_pct'].corr(sub['return_points_won_pct']):.2f}",
    )
axes[1].legend(title="surface")
axes[1].set_title("By surface (regression lines; observational)")
axes[1].set_xlabel("serve_win_pct (observed)")
axes[1].set_ylabel("return_points_won_pct (observed)")
axes[1].set_xticks(tick_rates)
axes[1].set_yticks(tick_rates)
fig.tight_layout(rect=(0, 0, 0.97, 0.92))

## Player profiles

One row per player. Bio fields come from `bronze_profiles`; the career win rate is
aggregated from `silver` (player-match rows) and joined by `player_id` only - names
are never used and were excluded at load. A few notes:

- **Grain and join**: the profile frames are each one row per player; the win rate
  is aggregated one row per player and joined on `player_id` only. Every chart
  declares this grain via `GRAINS["profiles"]`.
- **Coverage**: bio fields are sparse - handedness 64%, height 37%, turned_pro 21%
  of 7,619 profile rows. Population is inspected first; charts exist only for
  materially populated fields, and sparse fields are reported, not charted.
- **Outcome threshold**: win-rate comparisons require at least
  `MIN_WINRATE_MATCHES = 20` career matches per player, with sample sizes shown.
- **Interpretation**: all relationships are observational, never causal.


What to look for: whether handedness, height, or years professional relate to observed career win rate among players with enough matches (>= 20), keeping in mind that most bio fields are only partially recorded.

In [ ]:
# Grain: one row per player. Career win rate aggregated from silver player-match
# rows (player perspective) and joined to bronze bio fields on player_id only.
MIN_WINRATE_MATCHES = 20

player_winrate = (
    silver.groupby("player_id")["match_won"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "career_win_rate", "count": "career_matches"})
)
profiles = bronze_profiles.merge(player_winrate, on="player_id", how="left")

# Years professional relative to the latest match year in the snapshot (data-relative).
REF_YEAR = int(gold["match_date"].dt.year.max())
profiles["years_pro"] = REF_YEAR - profiles["turned_pro"]

GRAINS["profiles"] = profiles
print(f"profiles: {len(profiles):,} rows, one per player (joined on player_id only)")
print(
    f"players with a career win rate: {profiles['career_win_rate'].notna().sum():,} "
    f"({profiles['career_matches'].fillna(0).ge(MIN_WINRATE_MATCHES).sum():,} "
    f"above the >= {MIN_WINRATE_MATCHES}-match threshold)"
)

In [ ]:
# Profile population first: which fields are materially covered. Bio fields are
# sparse (handedness 64%, height 37%, turned_pro 21% of 7,619 rows); gold profile
# serve metrics are mostly missing because serve stats were unrecorded pre-1990.
bio_coverage = (
    bronze_profiles[["handedness", "height", "weight", "birthdate", "turned_pro"]]
    .notna()
    .mean()
    .sort_values(ascending=False)
)
print("bio field coverage (share of profile rows):")
print(bio_coverage.to_string())

gold_coverage = (
    gold_profiles[
        ["first_serve_in_pct", "return_points_won_pct", "current_rank", "career_win_rate"]
    ]
    .notna()
    .mean()
)
print("\ngold profile metric coverage:")
print(gold_coverage.to_string())
print(
    "\nChartable: handedness (64%), height (37%), years professional (21%). "
    "Skipped: weight and birthdate add no distinct information beyond height / "
    "years professional; sparse gold serve metrics are already reported above."
)

In [ ]:
# Grain: profiles (one row per player). Counts use all profile rows; the win-rate
# comparison applies the >= MIN_WINRATE_MATCHES threshold and annotates n per bar.
# Ambidextrous (A) qualified players number 3, too few for a rate, so the
# comparison keeps right/left only while the count panel shows all categories.
df = GRAINS["profiles"]
hand_counts = (
    df["handedness"].fillna("missing").value_counts().reindex(["R", "L", "A", "missing"]).fillna(0)
)

q = df[df["career_matches"] >= MIN_WINRATE_MATCHES]
q = q[q["handedness"].isin(["R", "L"])]
agg = (
    q.groupby("handedness", observed=True)["career_win_rate"]
    .agg(["mean", "count"])
    .reindex(["R", "L"])
)
overall = q["career_win_rate"].mean()
print(
    f"right/left-handed win-rate comparison: n={len(q):,} players "
    f"(>= {MIN_WINRATE_MATCHES} matches each)"
)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
sns.barplot(x=hand_counts.index, y=hand_counts.values, ax=axes[0])
for bar, n in zip(axes[0].patches, hand_counts.values, strict=True):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{n:,}",
        ha="center",
        fontsize=8,
    )
axes[0].set_title(f"Players by handedness (n={len(df):,} profile rows)")
axes[0].set_xlabel("handedness")
axes[0].set_ylabel("players")

sns.barplot(x=agg.index, y=agg["mean"], ax=axes[1])
axes[1].axhline(overall, color="gray", ls="--", lw=1)
for bar, n in zip(axes[1].patches, agg["count"], strict=True):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.004,
        f"n={n:,}",
        ha="center",
        fontsize=8,
    )
axes[1].set_ylim(0, 0.5)
axes[1].set_title(f"Career win rate by handedness (>= {MIN_WINRATE_MATCHES} matches)")
axes[1].set_xlabel("handedness")
axes[1].set_ylabel("career win rate (observed)")
fig.suptitle(
    "Handedness - one row per player, joined on player_id (observational)",
    y=0.98,
)
fig.tight_layout(rect=(0, 0, 1, 0.88))

In [ ]:
# Grain: profiles (one row per player). Height is recorded for 37% of profile rows;
# the histogram shows exactly that recorded subset (n shown). The win-rate
# relationship applies the >= MIN_WINRATE_MATCHES threshold. Observational.
df = GRAINS["profiles"]
hgt = df["height"].dropna()
q = df[(df["career_matches"] >= MIN_WINRATE_MATCHES) & df["height"].notna()]
r = q["height"].corr(q["career_win_rate"])
print(f"height recorded: {len(hgt):,} players; win-rate subset: n={len(q):,}, r={r:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
sns.histplot(hgt, bins=30, ax=axes[0])
axes[0].set_title(f"Player height (n={len(hgt):,} of {len(df):,} profile rows covered)")
axes[0].set_xlabel("height (cm)")
axes[0].set_ylabel("players")

sns.regplot(
    data=q,
    x="height",
    y="career_win_rate",
    scatter_kws={"s": 14, "alpha": 0.35},
    line_kws={"color": "crimson"},
    ax=axes[1],
)
axes[1].set_title(f"Height vs career win rate (n={len(q):,}, r={r:.2f})")
axes[1].set_xlabel("height (cm)")
axes[1].set_ylabel("career win rate (observed)")
fig.suptitle(
    f"Height - one row per player, >= {MIN_WINRATE_MATCHES} matches for the rate (observational)",
    y=0.98,
)
fig.tight_layout(rect=(0, 0, 1, 0.88))

In [ ]:
# Grain: profiles (one row per player). turned_pro is recorded for 21% of profile
# rows, so years_pro covers the recorded subset only (n shown) - a coverage note,
# not a chart of the missing. The win-rate relationship applies the match threshold.
df = GRAINS["profiles"]
yrs = df["years_pro"].dropna()
q = df[(df["career_matches"] >= MIN_WINRATE_MATCHES) & df["years_pro"].notna()]
r = q["years_pro"].corr(q["career_win_rate"])
print(
    f"turned_pro recorded: {len(yrs):,} players "
    f"({100 * len(yrs) / len(df):.0f}% of profile rows); win-rate subset: n={len(q):,}, r={r:.2f}"
)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
sns.histplot(yrs, bins=30, ax=axes[0])
axes[0].set_title(f"Years professional (n={len(yrs):,} players with turned_pro recorded)")
axes[0].set_xlabel("years professional")
axes[0].set_ylabel("players")

sns.regplot(
    data=q,
    x="years_pro",
    y="career_win_rate",
    scatter_kws={"s": 14, "alpha": 0.35},
    line_kws={"color": "crimson"},
    ax=axes[1],
)
axes[1].set_title(f"Years professional vs career win rate (n={len(q):,}, r={r:.2f})")
axes[1].set_xlabel("years professional")
axes[1].set_ylabel("career win rate (observed)")
fig.suptitle(
    f"Years professional - one row per player, >= {MIN_WINRATE_MATCHES} matches for the rate (observational)",
    y=0.98,
)
fig.tight_layout(rect=(0, 0, 1, 0.88))

## Player trends over time and all-player surface comparisons

Descriptive trends and population-wide comparisons. Every chart is
observational: it describes this data and does not establish causation.

- **Age by year** - `player_match` grain (silver), full frame: average player age
  at match time per year, with annual row counts below. Ages outside the
  plausible career [15, 50] are excluded and counted (same policy as the age
  histogram in section 3). Missing age is data availability, not a tennis
  pattern; only the 1968-1971 rows are meaningfully affected.
- **Height by year** - `player_year` grain: bronze height joined to silver
  strictly on `player_id` (names never used), deduplicated to one row per player
  per year so prolific players do not dominate the annual average. Yearly counts
  show the covered subset; height is recorded for only 37% of profile rows.
- **Surface comparison** - `player_match` grain, full frame, all four surfaces:
  observed serve/return rates (safe columns from the setup section) and recent
  workload, across the whole player population rather than the qualified
  specialization subset of section 4. Carpet is kept because it has data, but
  its serve stats cover 1991-2017 only, so its rates describe an older era;
  sparse coverage is labeled, not hidden.
- **Surface by year** - the same rate columns by year and surface, showing where
  each surface's coverage begins and ends.

What to look for: how average age and height drift across the snapshot, and
whether serve/return metrics separate by surface across all players - keeping in
mind that carpet's older era mixes surface and time.


In [ ]:
# Grain: player_match (silver), full frame, annual aggregate (never sampled).
# Average player age at match time by year. Ages outside the plausible playing
# career [15, 50] are dropped and counted (same policy as the age histogram in
# t3-03); the lower panel shows annual row counts so sparse early coverage is
# visible. Missing age is data availability, not a tennis pattern.
df = GRAINS["player_match"].copy()
df["year"] = df["match_date"].dt.year
valid = df[df["player_age"].between(15, 50)]
age_by_year = (
    valid.groupby("year")["player_age"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "avg_age", "count": "rows"})
)
age_missing = df.groupby("year")["player_age"].apply(lambda s: s.isna().mean())
print(
    f"age missing overall: {df['player_age'].isna().mean():.1%} of rows "
    f"(per-year share peaks at {age_missing.max():.1%} in the early years); "
    f"{len(df) - len(valid):,} rows with age outside [15, 50] excluded"
)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
sns.lineplot(x=age_by_year.index, y=age_by_year["avg_age"], ax=axes[0])
axes[0].set_title(
    "Average player age at match time by year (player_match grain, full frame, ages in [15, 50])"
)
axes[0].set_ylabel("average age (years)")
axes[0].set_xlabel("")
sns.barplot(x=age_by_year.index, y=age_by_year["rows"], ax=axes[1], color="gray")
axes[1].set_title(
    "Annual row count (coverage; missing age is availability, not a pattern)",
    fontsize=9,
)
axes[1].set_ylabel("player-match rows")
axes[1].set_xlabel("year")
fig.tight_layout()

In [ ]:
# Grain: player-year (one row per player per year). Bronze height is joined to
# silver strictly on player_id (names never used) and deduplicated to one row per
# player per year before averaging, so prolific players cannot dominate the
# annual mean. Yearly counts show the covered subset: height is recorded for 37%
# of profile rows, and 1967 contributes only 11 players (sparse coverage).
df = GRAINS["player_match"]
height_profile = bronze_profiles[["player_id", "height"]].dropna()
rows = df.merge(height_profile, on="player_id", how="inner")
rows["year"] = rows["match_date"].dt.year
player_year = rows.drop_duplicates(["player_id", "year"])
height_by_year = (
    player_year.groupby("year")["height"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "avg_height", "count": "players"})
)
print(
    f"height recorded for {len(height_profile):,} players; "
    f"{len(player_year):,} player-year rows after dedup; "
    f"players per year: {height_by_year['players'].min()}-{height_by_year['players'].max()}"
)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
sns.lineplot(x=height_by_year.index, y=height_by_year["avg_height"], ax=axes[0])
axes[0].set_title(
    "Average player height by year (player-year grain: bronze height joined on "
    "player_id, one row per player per year)"
)
axes[0].set_ylabel("average height (cm)")
axes[0].set_xlabel("")
sns.barplot(x=height_by_year.index, y=height_by_year["players"], ax=axes[1], color="gray")
axes[1].set_title(
    "Distinct players with recorded height per year (sparse = coverage, not a trend)",
    fontsize=9,
)
axes[1].set_ylabel("players")
axes[1].set_xlabel("year")
fig.tight_layout()

In [ ]:
# Grain: player_match, FULL frame - the whole player population, not the 8
# specialization players of t4-04. Observed serve/return rates from the safe
# t1-05 columns plus recent workload; rows without exposure or with out-of-range
# probability rates are dropped per panel and counted. Carpet is kept because it
# has data, but its serve stats cover 1991-2017 only (an older era), so sparse
# coverage is labeled rather than hidden. Observational only.
df = GRAINS["player_match"]
SURFACES_ALL = ["hard", "clay", "grass", "carpet"]
SURFACE_COMPARE_COLS = [
    "ace_rate",
    "first_serve_pct",
    "serve_win_pct",
    "return_points_won_pct",
    "break_points_saved_pct",
    "matches_30d_before",
]
PROB_RATE_COLS = set(SURFACE_COMPARE_COLS) - {"matches_30d_before"}

coverage = (
    df.groupby("surface", observed=True)
    .agg(rows=("surface", "size"), serve_exposed=("total_serve_points", lambda s: (s > 0).sum()))
    .reindex(SURFACES_ALL)
)
print("Per-surface coverage (full frame):")
print(coverage.to_string())

fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
fig.suptitle(
    "All-player surface comparison - player_match grain, full frame, all surfaces (observational)",
    y=0.98,
)
for ax, col in zip(axes.ravel(), SURFACE_COMPARE_COLS, strict=True):
    s = df.dropna(subset=[col])
    if col in PROB_RATE_COLS:
        s = s[s[col].between(0, 1)]
    dropped = len(df) - len(s)
    sns.violinplot(
        data=s,
        x="surface",
        y=col,
        order=SURFACES_ALL,
        hue="surface",
        hue_order=SURFACES_ALL,
        inner="box",
        legend=False,
        ax=ax,
    )
    n_per_surface = s.groupby("surface", observed=True).size().reindex(SURFACES_ALL).fillna(0)
    ymax = s[col].max()
    for xpos, n in enumerate(n_per_surface):
        ax.text(xpos, ymax * 1.02, f"n={int(n):,}", ha="center", fontsize=7)
    ax.set_ylim(0, ymax * 1.18)
    ax.set_title(f"{col} (total n={len(s):,}; {dropped:,} rows excluded)")
    ax.set_xlabel("")
    ax.set_ylabel("")
fig.tight_layout(rect=(0, 0, 1, 0.93))

In [ ]:
# Grain: player_match, full frame, annual aggregate. Mean observed rate by year
# and surface shows where each surface's coverage begins and ends: carpet's line
# stops in 2017 because its serve stats end there (sparse coverage, not a
# performance collapse). Observational only.
df = GRAINS["player_match"].copy()
df["year"] = df["match_date"].dt.year

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, col in zip(axes, ["ace_rate", "return_points_won_pct"], strict=True):
    s = df.dropna(subset=[col])
    s = s[s[col].between(0, 1)]
    trend = s.groupby(["year", "surface"], observed=True)[col].mean().reset_index()
    sns.lineplot(data=trend, x="year", y=col, hue="surface", hue_order=SURFACES_ALL, ax=ax)
    ax.set_title(f"Mean {col} by year and surface (carpet ends 2017: coverage, not a trend)")
    ax.set_xlabel("year")
    ax.set_ylabel(col)
fig.suptitle(
    "Surface rates over time - player_match grain, full frame (observational)",
    y=0.98,
)
fig.tight_layout(rect=(0, 0, 1, 0.9))

## Which columns later feed predictions

This last section takes a quick look at the columns that will be handed to a
prediction model in a later step. For each match, `gold` stores one row per
player, and the `_diff` columns compare the two players (for example
`rank_diff` is the player's rank minus the opponent's rank). A positive
association with `match_won` simply means "the player with the higher value
tended to win in this data".

The charts here are still observations: histograms of a few columns, win
probability by column value, and a correlation map showing which columns move
together. Correlation describes this data; it does not mean one column causes
another.

What to look for: which columns have the clearest relationship with winning, and which columns are so similar that they carry the same information.

In [ ]:
# Feature names and order come from src.features.columns, the single source of
# the feature schema used across the codebase.
from src.features.columns import (
    CONTEXT_COLS,
    DIFF_COLS,
    FEATURE_COLS,
    GRADIENT_COLS,
    H2H_COLS,
    PROFILE_COLS,
    RATE_EXPOSURE_COLS,
)

FAMILY_GROUPS = [
    ("diff", DIFF_COLS),
    ("elo_gradient", GRADIENT_COLS),
    ("rate_exposure", RATE_EXPOSURE_COLS),
    ("profile", PROFILE_COLS),
    ("h2h", H2H_COLS),
    ("context", CONTEXT_COLS),
]
gold_f = GRAINS["gold_directional"]
print(f"gold: {len(gold_f):,} directional rows (two per physical match)")

# Full unsampled last-10y directional window for aggregate charts in this section.
GRAINS["last_10y_gold"] = gold_10y

In [ ]:
# Grain: gold_directional, last_10y_gold_sample (fixed-seed dense sample) for
# per-row distributions. Diff features are anti-symmetric between the two
# reciprocal rows, so the mirrored rows only double the mass, never change shape.
df = GRAINS["last_10y_gold_sample"]
DIST_FEATURES = [
    "rank_diff",
    "elo_diff",
    "form_diff",
    "surface_form_diff",
    "streak_diff",
    "serve_win_pct_diff",
    "return_points_won_pct_diff",
    "ace_rate_diff",
    "first_serve_pct_diff",
    "h2h_advantage",
    "h2h_exposure",
    "days_since_last_match_diff",
]

fig, axes = plt.subplots(3, 4, figsize=(15.5, 9.5))
fig.suptitle(
    "Selected gold directional features - last-10y fixed-seed sample "
    "(diff features anti-symmetric around 0 by construction)",
    y=0.98,
)
for ax, col in zip(axes.ravel(), DIST_FEATURES, strict=True):
    s = df[col].dropna()
    sns.histplot(s, bins=50, ax=ax)
    ax.set_title(f"{col}\n(n={len(s):,})")
    ax.set_xlabel("")
    ax.set_ylabel("")
fig.tight_layout(rect=(0, 0, 1, 0.94))

In [ ]:
# Grain: gold_directional, full last-10y window (aggregate, never sampled).
# Observed win probability by feature bin for the strongest directional
# features; the outer ~5% tails per feature are clipped to keep bins dense
# (clipped rows counted in the title). Observational: realized outcomes in this
# snapshot, not model importance.
PROB_FEATURES = [
    "elo_diff",
    "rank_diff",
    "form_diff",
    "serve_win_pct_diff",
    "h2h_advantage",
    "days_since_last_match_diff",
]
df = GRAINS["last_10y_gold"]
overall = df["match_won"].mean()
print(f"overall directional win rate: {overall:.3f} (n={len(df):,} rows, full last-10y window)")

fig, axes = plt.subplots(2, 3, figsize=(16, 8.5))
for ax, col in zip(axes.ravel(), PROB_FEATURES, strict=True):
    s = df[["match_won", col]].dropna()
    q = np.quantile(s[col], [0.05, 0.95])
    kept = s[s[col].between(q[0], q[1])].copy()
    edges = np.unique(np.quantile(kept[col], np.linspace(0, 1, 9))).tolist()
    kept["bin"] = pd.cut(kept[col], bins=edges, include_lowest=True)
    agg = kept.groupby("bin", observed=True)["match_won"].agg(["mean", "count"])
    centers = [iv.mid for iv in agg.index]
    sns.barplot(x=centers, y=agg["mean"], ax=ax)
    ax.axhline(overall, color="gray", ls="--", lw=1)
    ax.set_title(f"{col} (n={len(kept):,}; {len(s) - len(kept):,} tail rows clipped)")
    ax.set_xlabel("")
    ax.set_ylabel("win probability")
    ax.tick_params(axis="x", rotation=45, labelsize=7)
fig.suptitle(
    "Observed win probability by feature bin - gold_directional, full last-10y window",
    y=0.98,
)
fig.tight_layout(rect=(0, 0, 1, 0.92))

In [ ]:
# Grain: gold_directional, FULL gold frame (never sampled). Pearson correlations
# over all FEATURE_COLS. Correlations among anti-symmetric diff features are
# unchanged by the mirrored reciprocal rows; context-vs-target correlations are
# structural zeros (the label balances within each match at this grain).
corr = gold_f[FEATURE_COLS].corr()

# Clustered full matrix: readable block structure with family coloring.
fam_palette = sns.color_palette("tab10", len(FAMILY_GROUPS))
family_of = {col: fam_palette[i] for i, (_, cols) in enumerate(FAMILY_GROUPS) for col in cols}
row_colors = pd.Series([family_of[c] for c in FEATURE_COLS], index=FEATURE_COLS)
g = sns.clustermap(
    corr,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    row_colors=row_colors,
    col_colors=row_colors,
    figsize=(13.5, 12.5),
    dendrogram_ratio=(0.12, 0.12),
)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), rotation=90, fontsize=6)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=6)
g.ax_heatmap.legend(
    handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in fam_palette],
    labels=[name for name, _ in FAMILY_GROUPS],
    title="family",
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
    fontsize=7,
)
g.fig.suptitle(
    "Feature correlation, clustered (full gold, gold_directional grain; family colors on row/col)",
    y=0.99,
)
assert g.cax is not None
g.cax.set_ylabel("Pearson r", fontsize=7)

# Restricted annotated view: the diff family is the largest block and where
# serve/return redundancy concentrates; only |r| >= 0.4 is shown.
d = corr.loc[DIFF_COLS, DIFF_COLS]
mask = d.abs() < 0.4
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(
    d,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.4,
    annot_kws={"fontsize": 5.5},
    ax=ax,
)
ax.set_title("diff-family correlations, |r| >= 0.4 shown (full gold, gold_directional grain)")
ax.tick_params(axis="x", rotation=90, labelsize=7)
ax.tick_params(axis="y", labelsize=7)
fig.tight_layout()
fig.axes[-1].set_ylabel("Pearson r")

In [ ]:
# Rankings over the FULL screened set (all FEATURE_COLS, full gold frame).
# Association is |Pearson r| with the target; redundancy is off-diagonal |r|
# between features. Both are descriptive associations, not model importance.
target_corr = gold_f[FEATURE_COLS].corrwith(gold_f["match_won"])
assoc = (
    pd.DataFrame({"feature": target_corr.index, "r": target_corr.round(4).values})
    .assign(abs_r=target_corr.abs().values)
    .sort_values("abs_r", ascending=False)
    .drop(columns="abs_r")
    .reset_index(drop=True)
)
print(f"Strongest absolute target associations (all {len(FEATURE_COLS)} FEATURE_COLS, full gold):")
print(assoc.to_string(index=False))
print()

pair_rows = [
    (a, b, corr.loc[a, b]) for i, a in enumerate(FEATURE_COLS) for b in FEATURE_COLS[i + 1 :]
]
pairs = (
    pd.DataFrame(pair_rows, columns=["feature_a", "feature_b", "r"])
    .assign(abs_r=lambda d: d["r"].abs())
    .sort_values("abs_r", ascending=False)
    .drop(columns="abs_r")
    .reset_index(drop=True)
)
print(
    f"Highest absolute feature-feature correlations (top 20 of {len(pairs):,} "
    "off-diagonal pairs, full gold):"
)
print(pairs.head(20).to_string(index=False))
print()
print(
    "Context features (is_clay/is_grass/is_hard/is_indoor, best_of, tournament_level, "
    "round_encoded) show |r| ~ 0 with match_won by construction: reciprocal rows balance "
    "the label within each physical match at the gold_directional grain. Their outcome "
    "relationships are examined at the physical_match grain in section 4, not here."
)

## Conclusion

Reading the sections in order: the data reaches back to the 1960s (with serve
statistics reliably recorded only from the 1990s on, and player biography only
partially filled in), career match volume is concentrated in a few very active
players, and over the last 10 years serve/return numbers separate by surface
while a better ranking or Elo goes hand in hand with winning. The final
section shows the columns that later feed predictions: the difference columns
carry most of the relationship with winning, and several of them move
together.

Three caveats apply to every chart above: event counts always use one row per
physical match (each match is never counted twice); dense plots use a fixed
random sample of the most recent 10 years while summaries use all rows; and
missing values are labeled as data availability, never as tennis patterns.
All comparisons are observational, not causal.